# Lab 03 — Entanglement and Bell States

Prepare the four maximally entangled Bell states, check their statevectors,
and measure their correlations in the $Z$ and $X$ bases. This entanglement
is the resource used in Labs 04 (teleportation) and 05 (superdense coding).

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

## Parameters

In [ ]:
BELL = "phi+"   # one of "phi+", "phi-", "psi+", "psi-"
SHOTS = 4096
SEED = 7

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, partial_trace
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

from common.qlab import save_circuit

backend = AerSimulator(seed_simulator=SEED)
s2 = 1 / np.sqrt(2)

# (Z on q0, X on q1) applied after H + CNOT, and the expected statevector
# in Qiskit order: index = q0 + 2*q1, i.e. amplitudes of |q1 q0> = 00, 01, 10, 11
BELL_STATES = {
    "phi+": ((0, 0), [s2, 0, 0, s2]),     # (|00> + |11>)/sqrt2
    "phi-": ((1, 0), [s2, 0, 0, -s2]),    # (|00> - |11>)/sqrt2
    "psi+": ((0, 1), [0, s2, s2, 0]),     # (|01> + |10>)/sqrt2
    "psi-": ((1, 1), [0, -s2, s2, 0]),    # (|01> - |10>)/sqrt2
}

if BELL not in BELL_STATES:
    raise ValueError(f"BELL must be one of {list(BELL_STATES)}.")

## Step 1 — Build the Bell-state circuit

In [ ]:
def bell_circuit(name, basis="Z"):
    (z, x), _ = BELL_STATES[name]

    q = QuantumRegister(2, "q")
    c = ClassicalRegister(2, "c")
    qc = QuantumCircuit(q, c)

    qc.h(q[0])
    qc.cx(q[0], q[1])
    qc.barrier(label="Entanglement Preparation")

    if x:
        qc.x(q[1])
    if z:
        qc.z(q[0])
    qc.barrier(label=f"Select |{name}>")

    if basis == "X":
        qc.h(q)
        qc.barrier(label="X-Basis Rotation")

    qc.measure(q, c)
    return qc


qc = bell_circuit(BELL)
save_circuit(qc, "circuit.png", scale=1.0);

## Step 2 — Statevector and reduced state
The pair is in a pure state, but each qubit **on its own** is maximally mixed ($I/2$):
all the information is in the correlations.

In [ ]:
state = Statevector(qc.remove_final_measurements(inplace=False))
print("Statevector (|q1 q0> = 00, 01, 10, 11):", np.round(state.data, 4))

rho_q0 = partial_trace(state, [1])
print("\nReduced density matrix of q0:\n", np.round(rho_q0.data, 4))
print("Purity Tr(rho^2) of q0:", round(float(np.real(rho_q0.purity())), 4), "(1 = pure, 0.5 = maximally mixed)")

## Step 3 — Correlations in the Z and X bases
The correlator is $E = P(\text{same}) - P(\text{different})$. It is the measured value of $\langle Z\otimes Z\rangle$ or $\langle X\otimes X\rangle$.

In [ ]:
def correlator(counts):
    shots = sum(counts.values())
    same = sum(n for k, n in counts.items() if k[0] == k[1])
    return (2 * same - shots) / shots


def measure(name, basis):
    compiled = transpile(bell_circuit(name, basis), backend)
    # A different seed per basis, so the two 50/50 runs are independent samples
    seed = SEED + (basis == "X")
    return backend.run(compiled, shots=SHOTS, seed_simulator=seed).result().get_counts()


z_counts = measure(BELL, "Z")
x_counts = measure(BELL, "X")
print(f"|{BELL}> Z basis:", z_counts, f"-> <ZZ> = {correlator(z_counts):+.3f}")
print(f"|{BELL}> X basis:", x_counts, f"-> <XX> = {correlator(x_counts):+.3f}")

plot_histogram([z_counts, x_counts], legend=["Z basis", "X basis"], title=f"|{BELL}>")

## Verification — all four Bell states

In [ ]:
EXPECTED_CORRELATORS = {"phi+": (+1, +1), "phi-": (+1, -1), "psi+": (-1, +1), "psi-": (-1, -1)}

print(f"{'state':<6} | {'<ZZ>':>6} | {'<XX>':>6} | purity(q0)")
for name, (_, vector) in BELL_STATES.items():
    sv = Statevector(bell_circuit(name).remove_final_measurements(inplace=False))
    zz, xx = correlator(measure(name, "Z")), correlator(measure(name, "X"))
    purity = float(np.real(partial_trace(sv, [1]).purity()))
    print(f"{name:<6} | {zz:+6.2f} | {xx:+6.2f} | {purity:.3f}")

    assert sv.equiv(Statevector(vector)), name
    assert (zz, xx) == EXPECTED_CORRELATORS[name], (name, zz, xx)
    assert np.isclose(purity, 0.5), (name, purity)

print("\nAll four Bell states verified: perfect correlations and maximally mixed subsystems.")

In [ ]:
qc.draw(output="mpl", scale=1.0)